# stride-zero-broadcast — ex1: diagnose zero-stride vs copy via .stride() + storage check

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `stride-zero-broadcast`. Running the final beacon cell reports progress against the `PyTorch: Zero-stride broadcasting` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Zero-stride broadcasting` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`stride-zero-broadcast`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "stride-zero-broadcast"
DD_SUBTOPIC = "PyTorch: Zero-stride broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Zero-stride broadcasting — quick refresher

When two tensors with mismatched shapes are combined elementwise, PyTorch broadcasts the smaller along the missing / size-1 axes. The implementation is dramatically cheaper than the user-facing semantics suggest: the smaller tensor's `.stride()` along the broadcast axis is set to **zero**, so the underlying memory is reused for every position along that axis — no copy, no allocation.

```python
x = torch.tensor([1., 2., 3.])           # shape (3,), strides (1,)
y = x.expand(4, 3)                       # shape (4, 3), strides (0, 1)
y.data_ptr() == x.data_ptr()             # True — same storage
```

**Three families to keep separate.**
1. **Implicit broadcast** — happens during arithmetic (`a + b`, `a * b`). No new tensor materialized.
2. **`.expand(shape)`** — creates a *view* with stride 0 on broadcast axes. Read-only in practice (writes alias all positions and usually error).
3. **`.repeat(...)`** / `einops.repeat` — actually *copies* data so every position has its own memory. Strides are all nonzero. Use this when you need to write into the broadcasted result.

**Why this matters.** A `(B, N, N)` attention mask broadcast from a `(N, N)` tensor adds zero memory cost. A `.repeat(B, 1, 1)` would multiply the mask footprint by `B` — a 4096-token transformer with batch 32 would balloon from 64 MB to 2 GB.

### Exercise 1 — diagnose zero-stride vs copy via .stride() + storage check

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Distinguish a zero-stride broadcast view from a true memory copy by reading `.stride()` and comparing `.data_ptr()` on both forms.
> Keywords: broadcast, stride, expand, repeat, memory
> ```

**KCs targeted:** `broadcast-stride-zero-readout`, `expand-vs-repeat-memory`

Implement `ex1_classify_broadcasts(x)` to characterize the two ways of replicating a 1-D vector into a 2-D matrix:

1. `x` is a 1-D `(N,)` float tensor (e.g. `t.tensor([1., 2., 3.])`).
2. Produce `expanded = x.expand(4, N)` — the *view* form.
3. Produce `repeated = x.repeat(4, 1)` — the *copy* form.
4. Return a dict with EXACTLY these keys:
   - `'expanded'`: the expanded tensor.
   - `'repeated'`: the repeated tensor.
   - `'expanded_stride'`: `tuple(expanded.stride())` — must be `(0, 1)` for an N-vector broadcast to `(4, N)`.
   - `'repeated_stride'`: `tuple(repeated.stride())` — must be `(N, 1)` (or whatever the contiguous (4, N) strides are).
   - `'expanded_shares_storage'`: bool — `expanded.data_ptr() == x.data_ptr()`. Should be `True`.
   - `'repeated_shares_storage'`: bool — `repeated.data_ptr() == x.data_ptr()`. Should be `False` (a copy was made).

The point: both produce a `(4, N)` tensor with the same values, but `.expand()` allocates ZERO new memory (stride 0 on the new axis points back at the original storage), while `.repeat()` copies `4 * N` floats into fresh memory.

**The print** at the end of your function should show the strides and data-ptr comparison so the caller sees the diagnostic.

In [ ]:
def ex1_classify_broadcasts(x: Tensor) -> dict:
    N = x.shape[0]
    expanded = x.expand(4, N)
    repeated = x.repeat(4, 1)
    out = {
        'expanded': expanded,
        'repeated': repeated,
        'expanded_stride': tuple(expanded.stride()),
        'repeated_stride': tuple(repeated.stride()),
        'expanded_shares_storage': expanded.data_ptr() == x.data_ptr(),
        'repeated_shares_storage': repeated.data_ptr() == x.data_ptr(),
    }
    print(f"  expanded.stride() = {out['expanded_stride']}  (zero on axis 0 → broadcast view)")
    print(f"  repeated.stride() = {out['repeated_stride']}  (all nonzero → fresh copy)")
    print(f"  expanded shares storage with x: {out['expanded_shares_storage']}")
    print(f"  repeated shares storage with x: {out['repeated_shares_storage']}")
    return out


<details><summary>Solution</summary>

```python
def ex1_classify_broadcasts(x: Tensor) -> dict:
    N = x.shape[0]
    expanded = x.expand(4, N)
    repeated = x.repeat(4, 1)
    out = {
        'expanded': expanded,
        'repeated': repeated,
        'expanded_stride': tuple(expanded.stride()),
        'repeated_stride': tuple(repeated.stride()),
        'expanded_shares_storage': expanded.data_ptr() == x.data_ptr(),
        'repeated_shares_storage': repeated.data_ptr() == x.data_ptr(),
    }
    print(f"  expanded.stride() = {out['expanded_stride']}  (zero on axis 0 → broadcast view)")
    print(f"  repeated.stride() = {out['repeated_stride']}  (all nonzero → fresh copy)")
    print(f"  expanded shares storage with x: {out['expanded_shares_storage']}")
    print(f"  repeated shares storage with x: {out['repeated_shares_storage']}")
    return out
```

**Why stride-0 is enough.** A tensor's `(i, j)` element is found at byte offset `i * stride[0] + j * stride[1]` from the storage base. If `stride[0] == 0`, every row index maps to the same offset — the underlying storage only needs `N` floats to serve a virtual `(4, N)` tensor.

**Why you usually can't write to an expanded view.** PyTorch blocks in-place writes through stride-0 axes precisely because the write would alias every position simultaneously, which is almost never the user's intent. `expanded[2, 1] = 99` raises `RuntimeError: unsupported operation: more than one element of the written-to tensor refers to a single memory location`.

**The 32-byte cost vs the 16-KB cost.** For `x` of length 4096 broadcast to `(32, 4096)`: `.expand()` uses 16 KB total (unchanged from `x`), `.repeat()` uses 512 KB. In a transformer with batch-broadcast attention masks, that gap compounds across every layer.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()